<a href="https://colab.research.google.com/github/SaharMian/recsys-letterboxd/blob/main/twotower.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import torch

BASE = '/content/drive/MyDrive/recsys/'
SEED, LIKE, K = 42, 4.0, 10
device = 'cuda'
print("GPU:", torch.cuda.is_available())

data = pd.read_parquet(BASE + 'split_3000.parquet')
data['u'] = data['user_name'].cat.codes
data['i'] = data['film_id'].cat.codes
n_users, n_films = data['u'].nunique(), data['i'].nunique()

train = data[data['split'] == 'train']
val   = data[data['split'] == 'val']
test  = data[data['split'] == 'test']
seen_train = train.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def relevant_sets(df):
    liked = df[df['rating'] >= LIKE]
    return liked.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def evaluate(recommend_fn, eval_df, seen):
    relevant = relevant_sets(eval_df)
    ndcgs, recalls, recommended = [], [], set()
    for user, rel in relevant.items():
        recs = recommend_fn(user, seen.get(user, set()), K)
        recommended.update(recs)
        hits = [1 if f in rel else 0 for f in recs]
        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
        ndcgs.append(dcg / idcg)
        recalls.append(sum(hits) / len(rel))
    return {'NDCG@10': np.mean(ndcgs), 'Recall@10': np.mean(recalls),
            'Täckning': len(recommended) / data['film_id'].nunique()}

pop_rank = train[train['rating'] >= LIKE]['film_id'].value_counts().index.tolist()
def recommend_pop(user, seen, k):
    out = []
    for f in pop_rank:
        if f not in seen:
            out.append(f)
            if len(out) == k:
                break
    return out

print(evaluate(recommend_pop, val, seen_train))

Mounted at /content/drive
GPU: True
{'NDCG@10': np.float64(0.18405938695253699), 'Recall@10': np.float64(0.03855764705308532), 'Täckning': 0.001364633254812769}


***filmmatcning***

In [4]:
import os, re

films = pd.read_csv(BASE + 'films.csv', engine='python', on_bad_lines='skip')
print("title.basics.tsv finns på Drive:", os.path.exists(BASE + 'title.basics.tsv'))

def norm(s):
    return re.sub(r'[^a-z0-9]+', '', str(s).lower())

def make_key(title, year):
    return title.map(norm) + '_' + year.astype(int).astype(str)

# Våra 61 555 filmer med titel och år
sample_films = pd.DataFrame({'film_id': data['film_id'].cat.categories})
sample_films = sample_films.merge(films[['film_id', 'film_name', 'year']], on='film_id', how='left')
sample_films['key'] = make_key(sample_films['film_name'], sample_films['year'].fillna(0))

# IMDb: bara filmtyper, med titel, år och genrer
imdb = pd.read_csv(BASE + 'title.basics.tsv', sep='\t',
                   usecols=['titleType', 'primaryTitle', 'startYear', 'genres'],
                   dtype=str, na_values='\\N')
imdb = imdb[imdb['titleType'].isin(['movie', 'tvMovie', 'short', 'video'])].dropna(subset=['startYear'])
imdb['key'] = make_key(imdb['primaryTitle'], imdb['startYear'])
imdb = imdb.drop_duplicates('key')[['key', 'genres']]

meta = sample_films.merge(imdb, on='key', how='left')
print(f"Filmer: {len(meta):,}")
print(f"Matchade med genrer: {meta['genres'].notna().mean():.1%}")
print("\nExempel på matchade:")
print(meta[meta['genres'].notna()].sample(5, random_state=SEED)[['film_name', 'year', 'genres']])
print("\nExempel på omatchade:")
print(meta[meta['genres'].isna()].sample(5, random_state=SEED)[['film_name', 'year']])

title.basics.tsv finns på Drive: True
Filmer: 61,555
Matchade med genrer: 85.1%

Exempel på matchade:
                  film_name    year                       genres
43582         Love You More  2008.0                  Drama,Short
15717     Chasing Happiness  2019.0  Biography,Documentary,Music
15486                Jekyll  2007.0       Horror,Sci-Fi,Thriller
48611        Borley Rectory  2017.0       Animation,Drama,Horror
57749  Harold's Going Stiff  2011.0                Comedy,Horror

Exempel på omatchade:
                        film_name    year
25048                       Se7en  1995.0
58890                Jungleground  1995.0
57807  Iria: Zeiram the Animation  1994.0
36998             Travolta and Me  1994.0
31211              The Gunfighter  2014.0


***vektorer åt ena tower***

In [5]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

def make_text(row):
    title = row['film_name'] if pd.notna(row['film_name']) else row['film_id'].replace('-', ' ')
    year = f" ({int(row['year'])})" if pd.notna(row['year']) else ""
    genres = f". Genres: {row['genres'].replace(',', ', ')}" if pd.notna(row['genres']) else ""
    return f"{title}{year}{genres}"

meta['text'] = meta.apply(make_text, axis=1)
print(meta['text'].head(3).tolist())

st_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
film_emb = st_model.encode(meta['text'].tolist(), batch_size=256,
                           show_progress_bar=True, normalize_embeddings=True)
print(film_emb.shape)

np.save(BASE + 'film_text_emb.npy', film_emb)

# Sanity check: vilka filmer ligger närmast Spider-Man 2 i vektorrymden?
q = meta.index[meta['film_id'] == 'spider-man-2']
q = q[0] if len(q) else 0
sims = film_emb @ film_emb[q]
top = np.argsort(-sims)[:6]
print("\nNärmaste grannar till:", meta.loc[q, 'text'])
print(meta.loc[top[1:], 'text'].tolist())

['008: Operation Exterminate (1965). Genres: Action, Drama', '0.0MHz (2019). Genres: Horror', '0.5 mm (2014). Genres: Drama']


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/241 [00:00<?, ?it/s]

(61555, 384)

Närmaste grannar till: Spider-Man 2 (2004). Genres: Action, Adventure, Sci-Fi
['Spider-Man 3 (2007). Genres: Action, Adventure, Sci-Fi', 'Spider-Man (2002). Genres: Action, Adventure, Sci-Fi', 'The Amazing Spider-Man 2 (2014). Genres: Action, Adventure, Sci-Fi', 'Spider (2007). Genres: Action, Drama, Short', 'The Amazing Spider-Man (2012). Genres: Action, Sci-Fi']


***träningsdata & profil***

In [6]:
film_emb = np.load(BASE + 'film_text_emb.npy')
film_text = torch.tensor(film_emb, dtype=torch.float32, device=device)

pos = train[train['rating'] >= LIKE]
pos_u = torch.tensor(pos['u'].values, dtype=torch.long, device=device)
pos_i = torch.tensor(pos['i'].values, dtype=torch.long, device=device)

seen_mask = torch.zeros(n_users, n_films, dtype=torch.bool, device=device)
seen_mask[torch.tensor(train['u'].values, dtype=torch.long, device=device),
          torch.tensor(train['i'].values, dtype=torch.long, device=device)] = True

def sample_negatives(u, n_neg=4):
    u = u.repeat_interleave(n_neg)
    neg = torch.randint(0, n_films, u.shape, device=device)
    bad = seen_mask[u, neg]
    while bad.any():
        neg[bad] = torch.randint(0, n_films, (int(bad.sum()),), device=device)
        bad = seen_mask[u, neg]
    return u, neg

# Användarprofil: medelvärdet av textvektorerna för filmer användaren gillat i train
user_profile = torch.zeros(n_users, film_text.shape[1], device=device)
user_profile.index_add_(0, pos_u, film_text[pos_i])
likes_per_user = torch.bincount(pos_u, minlength=n_users)
user_profile = user_profile / likes_per_user.clamp(min=1).unsqueeze(1)

print("film_text:", tuple(film_text.shape), "| user_profile:", tuple(user_profile.shape))
print("Positiva exempel:", f"{len(pos_u):,}", "| användare utan gillade filmer i train:", int((likes_per_user == 0).sum()))

film_text: (61555, 384) | user_profile: (3000, 384)
Positiva exempel: 1,333,735 | användare utan gillade filmer i train: 0


***Modellen och träningen***

In [7]:
import torch.nn as nn, copy

class TwoTower(nn.Module):
    def __init__(self, film_text, user_profile, d_id=32, d_out=64, p_id_drop=0.3):
        super().__init__()
        n_films, d_text = film_text.shape
        n_users = user_profile.shape[0]
        self.register_buffer('film_text', film_text, persistent=False)
        self.register_buffer('user_profile', user_profile, persistent=False)
        self.user_id = nn.Embedding(n_users, d_id)
        self.film_id = nn.Embedding(n_films, d_id)
        nn.init.normal_(self.user_id.weight, std=0.01)
        nn.init.normal_(self.film_id.weight, std=0.01)
        self.user_mlp = nn.Sequential(nn.Linear(d_text + d_id, 128), nn.ReLU(), nn.Linear(128, d_out))
        self.film_mlp = nn.Sequential(nn.Linear(d_text + d_id, 128), nn.ReLU(), nn.Linear(128, d_out))
        self.p = p_id_drop

    def user_vec(self, u):
        return self.user_mlp(torch.cat([self.user_profile[u], self.user_id(u)], dim=1))

    def film_vec(self, i):
        fid = self.film_id(i)
        if self.training and self.p > 0:
            fid = fid * (torch.rand(len(i), 1, device=i.device) > self.p).float()
        return self.film_mlp(torch.cat([self.film_text[i], fid], dim=1))

    def forward(self, u, i):
        return (self.user_vec(u) * self.film_vec(i)).sum(1)

def recommend_two_tower(model, exclude=('train',), k=K, chunk=500):
    ex = data[data['split'].isin(exclude)]
    ex_u = torch.tensor(ex['u'].values, dtype=torch.long, device=device)
    ex_i = torch.tensor(ex['i'].values, dtype=torch.long, device=device)
    users = data['user_name'].cat.categories
    films = data['film_id'].cat.categories
    model.eval()
    recs = {}
    with torch.no_grad():
        fv = model.film_vec(torch.arange(n_films, device=device))
        for start in range(0, n_users, chunk):
            end = min(start + chunk, n_users)
            u = torch.arange(start, end, device=device)
            scores = model.user_vec(u) @ fv.T
            sel = (ex_u >= start) & (ex_u < end)
            scores[ex_u[sel] - start, ex_i[sel]] = float('-inf')
            top = scores.topk(k, dim=1).indices.cpu().numpy()
            for row, uid in enumerate(range(start, end)):
                recs[users[uid]] = [films[j] for j in top[row]]
    return recs

N_NEG, BATCH, LR, EPOCHS = 4, 16384, 0.002, 10

torch.manual_seed(SEED)
tt = TwoTower(film_text, user_profile).to(device)
opt = torch.optim.Adam(tt.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
best_ndcg, best_state, best_epoch = -1, None, 0

for epoch in range(1, EPOCHS + 1):
    tt.train()
    u_neg, i_neg = sample_negatives(pos_u, N_NEG)
    u_all = torch.cat([pos_u, u_neg])
    i_all = torch.cat([pos_i, i_neg])
    y_all = torch.cat([torch.ones(len(pos_u)), torch.zeros(len(u_neg))]).to(device)
    perm = torch.randperm(len(y_all), device=device)

    total = 0.0
    for s in range(0, len(perm), BATCH):
        idx = perm[s:s + BATCH]
        loss = loss_fn(tt(u_all[idx], i_all[idx]), y_all[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)

    recs_tt = recommend_two_tower(tt)
    res = evaluate(lambda user, seen, kk: recs_tt[user], val, seen_train)
    print(f"Epok {epoch:>2}: loss {total / len(perm):.4f} | val NDCG {res['NDCG@10']:.4f}, "
          f"Recall {res['Recall@10']:.4f}, Täckning {res['Täckning']:.4f}")
    if res['NDCG@10'] > best_ndcg:
        best_ndcg, best_state, best_epoch = res['NDCG@10'], copy.deepcopy(tt.state_dict()), epoch

tt.load_state_dict(best_state)
torch.save(best_state, BASE + 'two_tower_model.pt')
print(f"Bästa epok: {best_epoch} (val NDCG {best_ndcg:.4f}), sparad till Drive")

# Lägg in i jämförelsetabellen
recs_tt = recommend_two_tower(tt)
r = evaluate(lambda user, seen, kk: recs_tt[user], val, seen_train)
tbl = pd.read_csv(BASE + 'results_val.csv', index_col=0)
tbl.loc[f'Two-tower (epok {best_epoch})'] = [round(r['NDCG@10'], 4), round(r['Recall@10'], 4), round(r['Täckning'], 4)]
tbl.to_csv(BASE + 'results_val.csv')
print(tbl)

Epok  1: loss 0.3175 | val NDCG 0.1692, Recall 0.0354, Täckning 0.0030
Epok  2: loss 0.2641 | val NDCG 0.1942, Recall 0.0419, Täckning 0.0053
Epok  3: loss 0.2420 | val NDCG 0.1904, Recall 0.0388, Täckning 0.0089
Epok  4: loss 0.2302 | val NDCG 0.1998, Recall 0.0423, Täckning 0.0083
Epok  5: loss 0.2215 | val NDCG 0.2107, Recall 0.0452, Täckning 0.0115
Epok  6: loss 0.2131 | val NDCG 0.2056, Recall 0.0439, Täckning 0.0145
Epok  7: loss 0.2068 | val NDCG 0.2103, Recall 0.0460, Täckning 0.0178
Epok  8: loss 0.2011 | val NDCG 0.2222, Recall 0.0491, Täckning 0.0217
Epok  9: loss 0.1958 | val NDCG 0.2183, Recall 0.0482, Täckning 0.0227
Epok 10: loss 0.1920 | val NDCG 0.2185, Recall 0.0477, Täckning 0.0277
Bästa epok: 8 (val NDCG 0.2222), sparad till Drive
                       NDCG@10  Recall@10  Täckning
Popularitet (gillade)   0.1841     0.0386    0.0014
SVD (min 0 betyg)       0.0086     0.0022    0.0323
SVD (min 1000 betyg)    0.1757     0.0365    0.0038
SVD (min 1500 betyg)    0.1929 

***3000 filmer ***

In [8]:
rng = np.random.default_rng(SEED)

# 3 000 "nya" filmer. De ska ha minst 20 betyg i train, så att de är vanliga filmer
train_counts = train['i'].value_counts()
eligible = train_counts[train_counts >= 20].index.to_numpy()
cold_idx = rng.choice(eligible, size=3000, replace=False)
train_cold = train[~train['i'].isin(cold_idx)]
print(f"Filmer att välja bland: {len(eligible):,}")
print(f"Train: {len(train):,} -> {len(train_cold):,} rader efter att de 3 000 filmerna tagits bort")

cold_mask = torch.zeros(n_films, dtype=torch.bool, device=device)
cold_mask[torch.tensor(cold_idx, dtype=torch.long, device=device)] = True

def relevant_cold(df):
    liked = df[(df['rating'] >= LIKE) & df['i'].isin(cold_idx)]
    return liked.groupby('u')['i'].apply(set).to_dict()

def evaluate_cold(score_fn, eval_df, chunk=500):
    rel = relevant_cold(eval_df)
    ndcgs, recalls = [], []
    for start in range(0, n_users, chunk):
        end = min(start + chunk, n_users)
        u = torch.arange(start, end, device=device)
        scores = score_fn(u).clone()
        scores[:, ~cold_mask] = float('-inf')          # bara de nya filmerna är kandidater
        top = scores.topk(K, dim=1).indices.cpu().numpy()
        for row, uid in enumerate(range(start, end)):
            if uid not in rel:
                continue
            r = rel[uid]
            hits = [1 if j in r else 0 for j in top[row]]
            dcg = sum(h / np.log2(p + 2) for p, h in enumerate(hits))
            idcg = sum(1 / np.log2(p + 2) for p in range(min(len(r), K)))
            ndcgs.append(dcg / idcg)
            recalls.append(sum(hits) / len(r))
    return {'NDCG@10': round(float(np.mean(ndcgs)), 4),
            'Recall@10': round(float(np.mean(recalls)), 4),
            'Användare': len(ndcgs)}

def make_profile(train_df):
    p = train_df[train_df['rating'] >= LIKE]
    pu = torch.tensor(p['u'].values, dtype=torch.long, device=device)
    pi = torch.tensor(p['i'].values, dtype=torch.long, device=device)
    prof = torch.zeros(n_users, film_text.shape[1], device=device)
    prof.index_add_(0, pu, film_text[pi])
    return prof / torch.bincount(pu, minlength=n_users).clamp(min=1).unsqueeze(1), pu, pi

profile_cold, pos_u_c, pos_i_c = make_profile(train_cold)

torch.manual_seed(SEED)
print("Slump:       ", evaluate_cold(lambda u: torch.rand(len(u), n_films, device=device), val))
print("Innehåll:    ", evaluate_cold(lambda u: profile_cold[u] @ film_text.T, val))

Filmer att välja bland: 22,312
Train: 3,552,662 -> 3,124,271 rader efter att de 3 000 filmerna tagits bort
Slump:        {'NDCG@10': 0.0029, 'Recall@10': 0.0029, 'Användare': 2853}
Innehåll:     {'NDCG@10': 0.0074, 'Recall@10': 0.0082, 'Användare': 2853}


***Träna om two-tower utan de 3 000 filmerna***

In [9]:
# Sanningstabell för train_cold. De nya filmerna får aldrig dras som negativa exempel
seen_mask_c = torch.zeros(n_users, n_films, dtype=torch.bool, device=device)
seen_mask_c[torch.tensor(train_cold['u'].values, dtype=torch.long, device=device),
            torch.tensor(train_cold['i'].values, dtype=torch.long, device=device)] = True
seen_mask_c[:, cold_mask] = True

def sample_neg(u, mask, n_neg=4):
    u = u.repeat_interleave(n_neg)
    neg = torch.randint(0, n_films, u.shape, device=device)
    bad = mask[u, neg]
    while bad.any():
        neg[bad] = torch.randint(0, n_films, (int(bad.sum()),), device=device)
        bad = mask[u, neg]
    return u, neg

def train_two_tower(pos_u, pos_i, profile, mask, epochs=8):
    torch.manual_seed(SEED)
    m = TwoTower(film_text, profile).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(1, epochs + 1):
        m.train()
        u_neg, i_neg = sample_neg(pos_u, mask, N_NEG)
        u_all, i_all = torch.cat([pos_u, u_neg]), torch.cat([pos_i, i_neg])
        y_all = torch.cat([torch.ones(len(pos_u)), torch.zeros(len(u_neg))]).to(device)
        perm = torch.randperm(len(y_all), device=device)
        total = 0.0
        for s in range(0, len(perm), BATCH):
            idx = perm[s:s + BATCH]
            loss = loss_fn(m(u_all[idx], i_all[idx]), y_all[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        print(f"  epok {epoch}: loss {total / len(perm):.4f}")
    return m

def tt_score_fn(m):
    m.eval()
    with torch.no_grad():
        fv = m.film_vec(torch.arange(n_films, device=device))
    def fn(u):
        with torch.no_grad():
            return m.user_vec(u) @ fv.T
    return fn

tt_cold = train_two_tower(pos_u_c, pos_i_c, profile_cold, seen_mask_c)

torch.manual_seed(SEED)
cold_results = {
    'Slump': evaluate_cold(lambda u: torch.rand(len(u), n_films, device=device), val),
    'Innehåll (ingen träning)': evaluate_cold(lambda u: profile_cold[u] @ film_text.T, val),
    'Two-tower (text + ID)': evaluate_cold(tt_score_fn(tt_cold), val),
}
cold_tbl = pd.DataFrame(cold_results).T
print(cold_tbl)
cold_tbl.to_csv(BASE + 'results_cold_val.csv')

  epok 1: loss 0.3203
  epok 2: loss 0.2633
  epok 3: loss 0.2436
  epok 4: loss 0.2286
  epok 5: loss 0.2189
  epok 6: loss 0.2108
  epok 7: loss 0.2042
  epok 8: loss 0.1992
                          NDCG@10  Recall@10  Användare
Slump                      0.0029     0.0029     2853.0
Innehåll (ingen träning)   0.0074     0.0082     2853.0
Two-tower (text + ID)      0.0267     0.0320     2853.0


***kontroll***

In [10]:
# 1. Opersonlig variant av den tränade modellen: samma rankning åt alla
tt_cold.eval()
with torch.no_grad():
    fv = tt_cold.film_vec(torch.arange(n_films, device=device))
    mean_u = tt_cold.user_vec(torch.arange(n_users, device=device)).mean(0, keepdim=True)
    same_for_all = mean_u @ fv.T
cold_results['Two-tower, opersonlig (samma lista åt alla)'] = evaluate_cold(
    lambda u: same_for_all.repeat(len(u), 1), val)

# 2. Bara ID, ingen text (vi nollar textvektorerna tillfälligt)
film_text_real = film_text
film_text = torch.zeros_like(film_text_real)
tt_id = train_two_tower(pos_u_c, pos_i_c, torch.zeros_like(profile_cold), seen_mask_c)
film_text = film_text_real
cold_results['Two-tower, bara ID (ingen text)'] = evaluate_cold(tt_score_fn(tt_id), val)

cold_tbl = pd.DataFrame(cold_results).T
print(cold_tbl)
cold_tbl.to_csv(BASE + 'results_cold_val.csv')

  epok 1: loss 0.3390
  epok 2: loss 0.3100
  epok 3: loss 0.3034
  epok 4: loss 0.2941
  epok 5: loss 0.2894
  epok 6: loss 0.2853
  epok 7: loss 0.2808
  epok 8: loss 0.2762
                                             NDCG@10  Recall@10  Användare
Slump                                         0.0029     0.0029     2853.0
Innehåll (ingen träning)                      0.0074     0.0082     2853.0
Two-tower (text + ID)                         0.0267     0.0320     2853.0
Two-tower, opersonlig (samma lista åt alla)   0.0200     0.0216     2853.0
Two-tower, bara ID (ingen text)               0.0031     0.0034     2853.0


In [11]:
import json

users = data['user_name'].cat.categories
pseudo = {u: f"User {k + 1}" for k, u in enumerate(users)}

# Rekommendationer för "nu": train och val räknas som redan sett
seen_tv = (data[data['split'].isin(['train', 'val'])]
           .groupby('user_name', observed=True)['film_id'].apply(set).to_dict())
recs_pop_app = {u: recommend_pop(u, seen_tv.get(u, set()), K) for u in users}
recs_tt_app = recommend_two_tower(tt, exclude=('train', 'val'))

def to_app(recs):
    return {pseudo[u]: [str(f) for f in fs] for u, fs in recs.items()}

# Några gillade filmer per användare, att visa i appen
liked = data[data['split'].isin(['train', 'val']) & (data['rating'] >= LIKE)]
hist = {pseudo[u]: g['film_id'].astype(str).sample(min(12, len(g)), random_state=SEED).tolist()
        for u, g in liked.groupby('user_name', observed=True)}

films_app = meta[['film_id', 'film_name', 'year', 'genres']].merge(
    films[['film_id', 'poster_url']], on='film_id', how='left')

os.makedirs(BASE + 'app_data', exist_ok=True)
json.dump(to_app(recs_pop_app), open(BASE + 'app_data/recs_popularity.json', 'w'))
json.dump(to_app(recs_tt_app), open(BASE + 'app_data/recs_two_tower.json', 'w'))
json.dump(hist, open(BASE + 'app_data/history.json', 'w'))
films_app.to_csv(BASE + 'app_data/films.csv', index=False)

print(list(hist.items())[0])
print(list(to_app(recs_tt_app).items())[0])
print(f"Filer sparade i {BASE}app_data/")

('User 1', ['the-first-wives-club', 'lady-bird', 'spider-man-across-the-spider-verse', 'batman-under-the-red-hood', 'maleficent', 'lost-ollie', 'mulan-2020', 'eyes-without-a-face', 'my-neighbor-totoro', 'encanto', 'the-many-adventures-of-winnie-the-pooh', 'you-again'])
('User 1', ['monsters-inc', 'avengers-endgame', 'barbie', 'the-dark-knight', 'the-super-mario-bros-movie', 'elf', 'inception', 'baby-driver', 'zodiac', 'shrek'])
Filer sparade i /content/drive/MyDrive/recsys/app_data/
